In [ ]:
import numpy as np
import torch 
from matplotlib import pyplot as plt
from utils import radonTransform, initial
from skimage.metrics import peak_signal_noise_ratio as PSNR
from skimage.transform import rescale, resize
from matplotlib.pyplot import imread, imsave
from scipy.io import loadmat
from MCMC.Fused_L_half.BPS_Gibbs import BPS_Gibbs
#from MCMC.BPS_Gibbs import BPS_Gibbs
#from MCMC.GS_Horseshoe import Gibbs_sampling
#from MCMC.Fused_L_half.GS_Fused_L_half import Gibbs_sampling
#from MCMC.PLD_sparse import proximal_langevin
#from MCMC.PCN_KL_sparse import PCN
#from MCMC.Fused_L_half.Empirical_Bayes import EB_Gibbs_BPS


In [ ]:
np.random.seed(5041294)
height, width = 128,128
angleNum = 64
img_mat = loadmat('GroundTruthReconstruction.mat')
img_true1 = img_mat['FBP1200'].astype(np.float32) / 65535.0
img_true2 = resize(img_true1, (height, width), mode='constant', anti_aliasing=True)
img_true= (img_true2-np.min(img_true2))/(np.max(img_true2)-np.min(img_true2))

A = radonTransform(angleNum, height, width).astype('float32')

x = img_true.reshape(-1, 1).astype('float32')
y_noise_free = A @ x
sigma = 0.01 * np.max(y_noise_free).astype('float32')
y = (y_noise_free + sigma * np.random.randn(*y_noise_free.shape)).astype('float32')

n = height * width
m = y.shape[0]

In [ ]:
plt.figure()
plt.imshow(x.reshape(height, width))
plt.colorbar(plt.imshow(x.reshape(height, width)))
plt.show()

In [ ]:
device1 = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [ ]:
x_init = initial(A,y,device1)

In [ ]:
hyper = [1,1,1,1]

#hyper = [1, 0.1, 1, 0.1]
#hyper = [10, 10]

#length_scale = 0.015
#nu = 2.5
#lambda_tv = 40
#beta = 0.0006

#hyper = [length_scale, nu, lambda_tv, beta]

In [ ]:
x_mean, x_std = BPS_Gibbs(x_init, torch.from_numpy(y).to(device1), torch.from_numpy(A.copy()).to(device1), sigma, hyper, gamma1 = 0, gamma2 = 1)
#x_mean,x_std, x_samples, excution_time = Gibbs_sampling(x_init, torch.from_numpy(y).to(device1), torch.from_numpy(A.copy()).to(device1), sigma, hyper, gamma1 = 1, gamma2 = 1)
#x_mean,x_std = proximal_langevin(x_init, torch.from_numpy(y).to(device1), torch.from_numpy(A.copy()).to(device1), sigma, hyper)
#x_mean,x_std, x_samples = PCN(torch.from_numpy(x).to(device1), torch.from_numpy(y).to(device1), torch.from_numpy(A.copy()).to(device1), sigma, hyper)

In [ ]:
#x_mean, x_std = EB_Gibbs_BPS(x_init, torch.from_numpy(y).to(device1), torch.from_numpy(A.copy()).to(device1), sigma, gamma1 = 0, gamma2 = 1)

In [ ]:
pixel = 128
plt.subplot(2, 1, 1)
plt.imshow((x_mean.view(pixel, pixel)).cpu().numpy(),cmap='gray')
plt.colorbar(plt.imshow((x_mean.view(pixel, pixel)).cpu().numpy(),cmap='gray'))
plt.subplot(2, 1, 2)
plt.imshow((x_std.view(pixel, pixel)).cpu().numpy(),cmap='gray')
plt.colorbar(plt.imshow((x_std.view(pixel, pixel)).cpu().numpy(),cmap='gray'))
plt.show()